In [9]:
import torch
import torch.nn as nn
import torch.optim as optim 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt 


In [10]:
# creating batches
BATCH_SIZE = 128

data_transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    transform=data_transform,
    download=True
)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)


In [11]:
# coding the VAe class

latent_dim = 64


class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=latent_dim):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)       # fc = fullyconnected
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)  # for mean
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)  # for log variance

        # Decoder
        self.fc2= nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)  # for reconstruction

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def encode(self, x):    
        h = self.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar



    def reparameterize(self, mu, logvar):    
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        z= mu + eps*std
        return z


    def decode(self, x):
        h = self.relu(self.fc2(z))
        out = self.sigmoid(self.fc3(h))
        return out
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_reconstructed = self.decode(z)
        return x_reconstructed, mu, logvar
      